In [1]:
import json
import os
from tqdm import tqdm
from langchain_community.document_loaders import PyPDFLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_community.embeddings import HuggingFaceEmbeddings
from langchain_community.vectorstores import Chroma

c:\Users\fujitsu\AppData\Local\Programs\Python\Python313\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
files = [
    "C:/Users/fujitsu/Documents/genen ai/cc tp rag/file1.pdf",
    "C:/Users/fujitsu/Documents/genen ai/cc tp rag/file2.pdf",
    "C:/Users/fujitsu/Documents/genen ai/cc tp rag/file3.pdf"
]

documents = []

for file in files:
    loader = PyPDFLoader(file)
    docs = loader.load()
    documents.extend(docs)

print("Nombre de documents chargés :", len(documents))

Nombre de documents chargés : 46


In [3]:
#chunking
text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=300,
    chunk_overlap=16,
    separators=["\n\n", "\n", ".", " ", ""] #nettoyage chunks 
)

chunks = text_splitter.split_documents(documents)

print("Nombre de chunks :", len(chunks))

Nombre de chunks : 287


In [4]:
#Embeddings
embedding_model = HuggingFaceEmbeddings(
    #model_name="sentence-transformers/all-MiniLM-L6-v2"
    model_name="thenlper/gte-small"
)
display(embedding_model)

C:\Users\fujitsu\AppData\Local\Temp\ipykernel_1268\3808828244.py:2: LangChainDeprecationWarning: The class `HuggingFaceEmbeddings` was deprecated in LangChain 0.2.2 and will be removed in 1.0. An updated version of the class exists in the `langchain-huggingface package and should be used instead. To use it run `pip install -U `langchain-huggingface` and import as `from `langchain_huggingface import HuggingFaceEmbeddings``.
  embedding_model = HuggingFaceEmbeddings(
Loading weights: 100%|██████████| 199/199 [00:00<00:00, 478.22it/s, Materializing param=pooler.dense.weight]                               
BertModel LOAD REPORT from: thenlper/gte-small
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


HuggingFaceEmbeddings(client=SentenceTransformer(
  (0): Transformer({'max_seq_length': 512, 'do_lower_case': False, 'architecture': 'BertModel'})
  (1): Pooling({'word_embedding_dimension': 384, 'pooling_mode_cls_token': False, 'pooling_mode_mean_tokens': True, 'pooling_mode_max_tokens': False, 'pooling_mode_mean_sqrt_len_tokens': False, 'pooling_mode_weightedmean_tokens': False, 'pooling_mode_lasttoken': False, 'include_prompt': True})
  (2): Normalize()
), model_name='thenlper/gte-small', cache_folder=None, model_kwargs={}, encode_kwargs={}, multi_process=False, show_progress=False)

In [5]:
#vector database
vectorstore = Chroma.from_documents(
    documents=chunks,
    embedding=embedding_model,
    persist_directory="./chroma_db"
)

retriever = vectorstore.as_retriever(
    search_type="mmr",
    search_kwargs={
        "k": 5,
        "fetch_k": 12,
        "lambda_mult": 0.7
    }
)
display(vectorstore)
display(retriever)

VectorStoreRetriever(tags=['Chroma', 'HuggingFaceEmbeddings'], vectorstore=<langchain_community.vectorstores.chroma.Chroma object at 0x000001F25A551FD0>, search_type='mmr', search_kwargs={'k': 5, 'fetch_k': 12, 'lambda_mult': 0.7})

In [6]:
from langchain_community.llms import Ollama

llm = Ollama(
    model="gemma3:270m",
    temperature=0
)

print("LLM loaded")

LLM loaded


C:\Users\fujitsu\AppData\Local\Temp\ipykernel_1268\892336112.py:3: LangChainDeprecationWarning: The class `Ollama` was deprecated in LangChain 0.3.1 and will be removed in 1.0.0. An updated version of the class exists in the `langchain-ollama package and should be used instead. To use it run `pip install -U `langchain-ollama` and import as `from `langchain_ollama import OllamaLLM``.
  llm = Ollama(


In [ ]:
from langchain_core.prompts import PromptTemplate

prompt_template = """
You are an advanced AI assistant specialized in answering questions using Retrieval-Augmented Generation (RAG).

Your task is to provide accurate, grounded, and reliable answers strictly based on the provided context.

---------------------
INSTRUCTIONS:
---------------------

1. Use ONLY the information available in the context.
2. Do NOT use any external knowledge, prior knowledge, or assumptions.
3. If the answer cannot be found explicitly or implicitly in the context, respond ONLY with:
   "I don't know".

4. Read the context carefully before answering.
5. Identify the most relevant parts of the context to answer the question.
6. If multiple pieces of information are relevant, combine them logically.

---------------------
ANSWER GUIDELINES:
---------------------

- Be clear, precise, and well-structured.
- Avoid unnecessary explanations or irrelevant details.
- Do not repeat the question in your answer.
- Use simple and understandable language.
- If the context contains technical information, explain it clearly.

- If applicable, structure your answer using:
  • bullet points
  • short paragraphs

- If the context includes steps or a process, present them in order.

---------------------
LANGUAGE:
---------------------

- Answer in the SAME language as the question.
- If the question is in French, answer in French.
- If the question is in English, answer in English.

---------------------
CONTEXT HANDLING:
---------------------

- Do not mention the existence of the context.
- Do not say phrases like "based on the context".
- Act as if the information is your own knowledge.

---------------------
SAFETY RULES:
---------------------

- Do not hallucinate or invent information.
- Do not guess if unsure.
- Prefer saying "I don't know" rather than giving a wrong answer.

---------------------
INPUT:
---------------------

Context:
{context}

Question:
{question}

---------------------
OUTPUT:
---------------------

Answer:
"""

PROMPT = PromptTemplate(
    template=prompt_template,
    input_variables=["context", "question"]
)

In [13]:
from langchain_core.runnables import RunnablePassthrough
from langchain_core.output_parsers import StrOutputParser

# 1. Configuration de la chaîne RAG moderne (Remplace RetrievalQA)
# Cette structure lie le retriever, le prompt et le LLM
rag_chain = (
    {"context": retriever, "question": RunnablePassthrough()}
    | PROMPT
    | llm
    | StrOutputParser()
)

print("✅ Chaîne RAG initialisée avec succès (Syntaxe LCEL)")

# 2. Test rapide (Optionnel - Tu peux décommenter pour tester une question)
# question_test = "Quelle est l'information principale du file1.pdf ?"
# reponse = rag_chain.invoke(question_test)
# print(f"Test Réponse : {reponse}")

✅ Chaîne RAG initialisée avec succès (Syntaxe LCEL)


In [14]:
with open("questionSource.json", "r") as f:
    true_answer = json.load(f)

print("Dataset chargé :", len(true_answer))

Dataset chargé : 3


In [ ]:
import json
from tqdm import tqdm

# Initialisation de la liste pour stocker les résultats
generated_answer = []

# On parcourt le dataset true_answer 
for doc in true_answer:
    new_doc = {
        # Utilisation de "file_name" pour correspondre à JSON source
        "file_name": doc.get("file_name", doc.get("document")), 
        "qa_pairs": []
    }
    
    print(f"\nTraitement du fichier : {new_doc['file_name']}")
    
    for qa in tqdm(doc["qa_pairs"], desc="Génération RAG"):
        question = qa["question"]
        
    
        response = rag_chain.invoke(question)
        
        new_doc["qa_pairs"].append({
            "question": question,
            "answer": response
        })
    
    generated_answer.append(new_doc)

# Étape 4 — Sauvegarde dans questionResults.json
with open("questionResults.json", "w", encoding="utf-8") as f:
    json.dump(generated_answer, f, indent=4, ensure_ascii=False)

print("\n✅ Étape 4 terminée : Le fichier 'questionResults.json' a été généré.")


Traitement du fichier : file1.pdf


Génération RAG: 100%|██████████| 10/10 [01:08<00:00,  6.87s/it]



Traitement du fichier : file2.pdf


Génération RAG: 100%|██████████| 10/10 [01:15<00:00,  7.51s/it]



Traitement du fichier : file3.pdf


Génération RAG: 100%|██████████| 10/10 [01:11<00:00,  7.19s/it]


✅ Étape 4 terminée : Le fichier 'questionResults.json' a été généré.


In [35]:
with open("questionResult.json", "w") as f:
    json.dump(generated_answer, f, indent=4)

print("Fichier sauvegardé ✅")

Fichier sauvegardé ✅


In [17]:
from sentence_transformers import SentenceTransformer
from sklearn.metrics.pairwise import cosine_similarity

# charger modèle embedding
model = SentenceTransformer("thenlper/gte-base")

def evaluer_score(true_answer, generated_answer):

    # transformer en vecteurs
    emb1 = model.encode([true_answer])
    emb2 = model.encode([generated_answer])

    # similarité cosinus
    sim = cosine_similarity(emb1, emb2)[0][0]

    # score sur 5
    if sim >= 0.9:
        score = 5
    elif sim >= 0.75:
        score = 4
    elif sim >= 0.6:
        score = 3
    elif sim >= 0.4:
        score = 2
    else:
        score = 1

    # pourcentage
    percent = sim * 100

    return score, percent, sim

c:\Users\fujitsu\AppData\Local\Programs\Python\Python313\Lib\site-packages\huggingface_hub\file_download.py:129: UserWarning: `huggingface_hub` cache-system uses symlinks by default to efficiently store duplicated files but your machine does not support them in C:\Users\fujitsu\.cache\huggingface\hub\models--thenlper--gte-base. Caching files will still work but in a degraded version that might require more space on your disk. This warning can be disabled by setting the `HF_HUB_DISABLE_SYMLINKS_WARNING` environment variable. For more details, see https://huggingface.co/docs/huggingface_hub/how-to-cache#limitations.
To support symlinks on Windows, you either need to activate Developer Mode or to run Python as an administrator. In order to activate developer mode, see this article: https://docs.microsoft.com/en-us/windows/apps/get-started/enable-your-device-for-development
  warnings.warn(message)
Loading weights: 100%|██████████| 199/199 [00:00<00:00, 611.21it/s, Materializing param=pool

In [18]:
scores = []
pourcentages = []
similarites = []

for d1, d2 in zip(true_answer, generated_answer):

    for qa1, qa2 in zip(d1["qa_pairs"], d2["qa_pairs"]):

        true_answer = qa1["answer"]
        generated_answer = qa2["answer"]

        score, percent, sim = evaluer_score(true_answer, generated_answer)

        scores.append(score)
        pourcentages.append(percent)
        similarites.append(sim)

# affichage
print("Scores individuels :", scores)
print("Pourcentages individuels :", pourcentages)

# moyenne
score_moyen = sum(scores) / len(scores)

# pourcentage total
pourcentage_total = (score_moyen / 5) * 100

print("Score moyen :", score_moyen)
print("Score total en % :", pourcentage_total)

Scores individuels : [5, 4, 5, 5, 4, 4, 4, 4, 3, 4, 5, 5, 5, 5, 5, 4, 5, 5, 5, 5, 5, 5, 5, 5, 4, 5, 5, 4, 4, 4]
Pourcentages individuels : [np.float64(93.63294426983656), np.float64(85.47126904931503), np.float64(90.38893079896636), np.float64(90.30537089964677), np.float64(77.29134226829643), np.float64(87.98371985167894), np.float64(88.99658690090038), np.float64(89.62057658772599), np.float64(73.1896624131169), np.float64(88.98858077868188), np.float64(95.08206011209161), np.float64(90.0046159660886), np.float64(90.19632361108921), np.float64(93.09459511847955), np.float64(93.73967233404169), np.float64(89.90387578197021), np.float64(92.86788933353333), np.float64(94.3420169815783), np.float64(91.74052282103843), np.float64(90.52051666420363), np.float64(95.54909209462461), np.float64(93.23136985010106), np.float64(90.95195514226609), np.float64(95.27440959563043), np.float64(86.3514374732), np.float64(96.31009743400546), np.float64(91.24522060287855), np.float64(87.26051291659364),